# Name That LLM — T4 capacity probe (NOT scientific transfer)

This notebook **only tests** whether the frozen Qwen3 24-layer / DCT-v3 / LoRA-r16 architecture can execute **one synthetic full-length triplet** with forward, backward, and an optimizer step on the assigned GPU.

- Works with a Tesla T4's approximately **14.56 GiB**; no arbitrary 15.00 GiB preflight cutoff.
- **No Kaggle credentials, competition data, heldout data, Public LB, model selection, or submission.**
- **No hyperparameter changes** and no checkpoint used in the scientific v3 evaluation. Any reported loss is meaningless for the competition (synthetic tokens/targets).
- No automatic switch to 4-bit/QLoRA, fewer layers, smaller batches, or CPU offload.
- Source is pinned to commit `b86e8cb68631a18a2a4d1342658ecdca740a3f2b`.
- Running may download many GiB of public model weights and fail if Colab runs out of memory, disk, or time. That failure is useful feasibility evidence, not a failure of the research hypothesis.

**Do not run the complete v1-v3 transfer** based on this alone: the frozen long training run currently lacks intermediate CUDA resume checkpoints. Only consider full training on a device/runtime with enough VRAM and duration.

In [ ]:
import torch, subprocess, json, shutil
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU"
p = torch.cuda.get_device_properties(0)
capacity_gib = p.total_memory / 1024**3
print(json.dumps({"gpu":p.name,"gpu_vram_gib":round(capacity_gib,2),"torch":torch.__version__,"cuda":torch.version.cuda, "free_disk_gib":round(shutil.disk_usage("/content").free/1024**3,2)},indent=2))
subprocess.run(["nvidia-smi"],check=True)

In [ ]:
import subprocess
from pathlib import Path
PINNED_SOURCE_COMMIT = "b86e8cb68631a18a2a4d1342658ecdca740a3f2b"
REPO = Path("/content/name-kaggle-t4-probe")
if not (REPO/".git").is_dir():
    subprocess.run(["git","init",str(REPO)],check=True)
    subprocess.run(["git","-C",str(REPO),"remote","add","origin","https://github.com/kolihhan/name-kaggle.git"],check=True)
    subprocess.run(["git","-C",str(REPO),"sparse-checkout","init","--cone"],check=True)
    subprocess.run(["git","-C",str(REPO),"sparse-checkout","set","scripts"],check=True)
    subprocess.run(["git","-C",str(REPO),"fetch","--depth=1","--filter=blob:none","origin",PINNED_SOURCE_COMMIT],check=True)
    subprocess.run(["git","-C",str(REPO),"checkout","--detach","FETCH_HEAD"],check=True)
observed = subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
assert observed == PINNED_SOURCE_COMMIT, observed
for secret_file in ["data/train.csv","data/test.csv","data/sample_submission.parquet"]:
    assert not (REPO/secret_file).exists(), secret_file
print("Pinned source verified; no competition data checked out.")

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,"-m","pip","install","-q",
    "transformers==4.57.1","peft==0.17.1","tokenizers==0.22.1",
    "huggingface-hub==0.36.0","safetensors"],check=True)
import transformers, peft
assert transformers.__version__ == "4.57.1"
assert peft.__version__ == "0.17.1"
print("Pinned dependencies:",transformers.__version__,peft.__version__)

In [ ]:
import gc, json, os, sys, time
os.chdir(REPO)
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
# Compatibility workaround copied from the frozen GPU worker: torchao is not used.
import importlib.util as importlib_util
_original_find_spec = importlib_util.find_spec
importlib_util.find_spec = lambda name,*args,**kwargs: None if name.split(".")[0]=="torchao" else _original_find_spec(name,*args,**kwargs)
import peft.import_utils
peft.import_utils.is_torchao_available = lambda *args,**kwargs: False

import torch.nn.functional as F
from torch import nn
from peft import LoraConfig, get_peft_model
from transformers import AutoConfig, AutoModel, AutoTokenizer
from scripts.dct_v3_cpu_feasibility import _download_truncated_snapshot, MODEL_ID, MODEL_REVISION
from scripts.run_two_stage_lora24_gate_a import gate_a_training_spec, center_triplet
from scripts.dct_temporal_pool import masked_dct_pool

t0 = time.perf_counter()
device = torch.device("cuda")
spec = gate_a_training_spec()
assert int(spec["keep_layers"])==24
assert int(spec["max_length_before_eos"])==447
assert list(spec["pooled_layers"])==[2,4,6,8,12,16,20,24]
assert int(spec["lora"]["r"])==16
root, download = _download_truncated_snapshot(Path("/content/qwen3-24layer-t4-probe"))
print("Model shards downloaded:",download["selected_shard_count"],"| disk GiB:",round(download["selected_shard_gib"],2),flush=True)
cfg = AutoConfig.from_pretrained(str(root),local_files_only=True)
assert cfg.model_type=="qwen3" and int(cfg.num_hidden_layers)==36 and int(cfg.hidden_size)==4096
cfg.num_hidden_layers=24
cfg.use_cache=False
if getattr(cfg,"layer_types",None) is not None: cfg.layer_types=list(cfg.layer_types[:24])
tokenizer=AutoTokenizer.from_pretrained(str(root),local_files_only=True)
torch.manual_seed(int(spec["seed"]))
try:
    base=AutoModel.from_pretrained(str(root),config=cfg,torch_dtype=torch.float16,
                                    local_files_only=True,low_cpu_mem_usage=True).to(device)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
    base.enable_input_require_grads()
    lora=spec["lora"]
    model=get_peft_model(base,LoraConfig(
        r=int(lora["r"]),lora_alpha=int(lora["alpha"]),
        lora_dropout=float(lora["dropout"]),bias=str(lora["bias"]),
        target_modules=list(lora["target_modules"])))
    for param in model.parameters():
        if param.requires_grad: param.data=param.data.float()
    layers=[int(v) for v in spec["pooled_layers"]]
    norms=nn.ModuleList([nn.LayerNorm(4096) for _ in layers]).to(device).float()
    plain=nn.Linear(4096*len(layers)*2,18).to(device).float()
    centered=nn.Linear(4096*len(layers)*2,18).to(device).float()
    dropout=nn.Dropout(float(spec["head_dropout"]))
    adapter_params=[p for p in model.parameters() if p.requires_grad]
    head_params=list(norms.parameters())+list(plain.parameters())+list(centered.parameters())
    optimizer=torch.optim.AdamW([
        {"params":adapter_params,"lr":float(spec["adapter_lr"])},
        {"params":head_params,"lr":float(spec["head_lr"])}
    ],weight_decay=float(spec["weight_decay"]))
    torch.cuda.reset_peak_memory_stats()
    print("Model loaded | allocated GiB:",round(torch.cuda.memory_allocated()/1024**3,3),flush=True)
    eos=int(tokenizer.eos_token_id)
    input_ids=torch.randint(10,int(cfg.vocab_size)-1,(3,447),device=device)
    input_ids=torch.cat([input_ids,torch.full((3,1),eos,device=device)],dim=1)
    mask=torch.ones_like(input_ids)
    target=torch.tensor([0,1,2],device=device)
    model.train(); norms.train(); plain.train(); centered.train()
    batch_start=time.perf_counter()
    with torch.autocast("cuda",dtype=torch.float16):
        hidden=model(input_ids=input_ids,attention_mask=mask,output_hidden_states=True,use_cache=False).hidden_states
    pooled=[]
    for layer,norm in zip(layers,norms):
        modes=masked_dct_pool(hidden[layer].float(),mask,mask_first_token=True,n_modes=2)
        pooled.extend([norm(modes[:,0]),norm(modes[:,1])])
    z=torch.cat(pooled,dim=-1)
    loss=F.cross_entropy(plain(dropout(z)),target)+F.cross_entropy(centered(dropout(center_triplet(z))),target)
    assert torch.isfinite(loss).item()
    # This is a SYNTHETIC memory probe. It does not represent a valid optimizer step of the frozen Stage-1 epoch.
    loss.backward()
    optimizer.step()
    torch.cuda.synchronize()
    results={
        "scope":"synthetic_t4_memory_probe_only",
        "scientific_decision_authorized":False,
        "official_rows_scored":0,
        "competition_test_rows_used":0,
        "public_lb_tuning":False,
        "gpu":torch.cuda.get_device_name(),
        "vram_gib":round(capacity_gib,3),
        "peak_allocated_gib":round(torch.cuda.max_memory_allocated()/1024**3,3),
        "peak_reserved_gib":round(torch.cuda.max_memory_reserved()/1024**3,3),
        "synthetic_single_triplet_step_seconds":round(time.perf_counter()-batch_start,2),
        "model_download_load_and_probe_seconds":round(time.perf_counter()-t0,2),
        "synthetic_forward_backward_success":True
    }
    path=Path("/content/dct_v3_t4_probe_result.json")
    path.write_text(json.dumps(results,indent=2))
    print(json.dumps(results,indent=2),flush=True)
except torch.cuda.OutOfMemoryError as e:
    print("T4 OUT OF MEMORY: frozen 24-layer/FP16/LoRA-r16 single-triplet probe did not fit.",flush=True)
    print("This is an infrastructure failure; v3 science remains unscored.",flush=True)
    print("Peak allocated GiB:",round(torch.cuda.max_memory_allocated()/1024**3,3),flush=True)
    raise
